# TP – Systèmes de recommandation

**Cours** : INF5063 – Machine learning : applications (G. Nollet, L. Benedetti)
**Auteur** : Alban Rouault

Objectif : concevoir et évaluer un système de recommandation de films par filtrage collaboratif
*user-based* sur le jeu de données Kaggle
[The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset).

## Imports et configuration

Toutes les dépendances sont déclarées dans `pyproject.toml` et gérées avec `uv`
(`uv run jupyter lab` pour lancer le notebook).

In [ ]:
%load_ext autoreload
%autoreload 2

import inspect
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from donnees import split_par_utilisateur, parts_validation_croisee   # séparations (donnees.py)
from modeles import RecoUserBased, RecoItemBased   # modèles (modeles.py)
from evaluation import predire_test, rmse_mae, evaluer_modele          # métriques (evaluation.py)

# Affichage des DataFrames
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

# Reproductibilité : graine de tous les tirages aléatoires (chaque cellule qui tire crée son propre générateur)
RANDOM_STATE = 42

# Emplacement des données
DATA_DIR = Path("Ressources/dataset")

## Exercice 1 – Chargement des données

### 1) Chargement des fichiers

> Récupérer sur Kaggle et charger les fichiers suivants du dataset : `movies_metadata.csv`, `ratings_small.csv` et `links_small.csv`.

In [ ]:
movies = pd.read_csv(DATA_DIR / "movies_metadata.csv", low_memory=False)
ratings = pd.read_csv(DATA_DIR / "ratings_small.csv")
links = pd.read_csv(DATA_DIR / "links_small.csv")

tables = {"movies_metadata": movies, "ratings_small": ratings, "links_small": links}
for name, df in tables.items():
    print(f"{name:<16} {df.shape[0]:>7,} lignes  x {df.shape[1]:>2} colonnes")

**Réponse.** Les trois fichiers viennent de
[The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset) (Kaggle) et sont
dans `Ressources/dataset/`. `movies_metadata.csv` est lu avec `low_memory=False` : quelques lignes
mal formées mélangent les types, on les traite à la question 5.

### 2) Aperçu des tables

> Afficher un aperçu de chaque table et vérifier leur compréhension.

In [ ]:
for name, df in tables.items():
    print(f"── {name} : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
    display(df.head(3))

In [ ]:
def resume_colonnes(df: pd.DataFrame) -> pd.DataFrame:
    """Type, remplissage et cardinalité de chaque colonne."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "% nuls": (df.isna().mean() * 100).round(1),
        "valeurs distinctes": df.nunique(),
        "exemple": df.iloc[0].astype(str).str.slice(0, 40),
    })

resume_colonnes(movies)

In [ ]:
print("Valeurs de rating :", sorted(ratings["rating"].unique().tolist()))
print("Période des avis  :",
      pd.to_datetime(ratings["timestamp"].min(), unit="s").date(), "->",
      pd.to_datetime(ratings["timestamp"].max(), unit="s").date())
print("Nuls dans ratings :", ratings.isna().sum().sum(), "| nuls dans links :", links.isna().sum().to_dict())

**Réponse.**

**`movies_metadata`** – une ligne par film du catalogue TMDB (45 466 lignes).

| Colonne | Contenu |
|---|---|
| `id` | identifiant TMDB du film (clé) |
| `imdb_id` | identifiant IMDB (`tt0114709`) |
| `title`, `original_title` | titre (anglais) et titre original |
| `overview`, `tagline` | synopsis et phrase d'accroche |
| `genres` | liste de genres, stockée comme texte JSON |
| `release_date` | date de sortie |
| `runtime` | durée en minutes |
| `budget`, `revenue` | budget et recettes mondiales en dollars (0 = inconnu) |
| `popularity` | score de popularité TMDB |
| `vote_average`, `vote_count` | note moyenne sur 10 et nombre de votes sur TMDB |
| `original_language`, `spoken_languages` | langue d'origine (code ISO) et langues parlées |
| `production_companies`, `production_countries` | sociétés et pays de production (texte JSON) |
| `belongs_to_collection` | saga du film (Toy Story Collection…), vide dans 90 % des cas |
| `status` | Released, Post Production, Rumored… |
| `adult`, `video` | film pour adultes ; vidéo hors salle (rares) |
| `homepage`, `poster_path` | site officiel et chemin de l'affiche |

**`ratings_small`** – une ligne par avis (100 004 lignes).

| Colonne | Contenu |
|---|---|
| `userId` | identifiant de l'utilisateur (MovieLens) |
| `movieId` | identifiant du film (MovieLens, **différent** de l'id TMDB) |
| `rating` | note de 0,5 à 5 par pas de 0,5 |
| `timestamp` | date de l'avis (secondes Unix, de 1995 à 2016) |

**`links_small`** – une ligne par film du jeu réduit (9 125 lignes).

| Colonne | Contenu |
|---|---|
| `movieId` | identifiant MovieLens (celui de `ratings_small`) |
| `imdbId` | identifiant IMDB |
| `tmdbId` | identifiant TMDB (celui de `movies_metadata`), manquant pour 13 films |

À retenir : `ratings_small` est propre (aucun nul), alors que `movies_metadata` a des colonnes très
peu remplies et des colonnes numériques lues comme du texte (`id`, `budget`, `popularity`) à cause de
lignes mal formées.

### 3) Effectifs

> Combien d'utilisateurs y a-t-il ? De films ? D'avis ?

In [ ]:
n_users = ratings["userId"].nunique()
n_movies_rated = ratings["movieId"].nunique()
n_ratings = len(ratings)
avis_par_user = ratings.groupby("userId").size()

pd.DataFrame({
    "valeur": [n_users, n_ratings, n_movies_rated, links["movieId"].nunique(), movies["id"].nunique(),
               f"{avis_par_user.min()} / {avis_par_user.median():.0f} / {avis_par_user.max()}",
               f"{n_ratings / (n_users * n_movies_rated):.2%}",
               f"{(ratings['rating'] >= 4).mean():.0%}", (ratings.groupby("movieId").size() == 1).sum()],
}, index=["utilisateurs", "avis", "films notés", "films dans links_small", "films dans movies_metadata",
          "avis par utilisateur (min / médiane / max)", "densité de la matrice utilisateurs x films",
          "part des avis >= 4", "films notés une seule fois"])

**Réponse.**

| | | |
|---|---|---|
| Utilisateurs | **671** | |
| Avis | **100 004** | |
| Films notés | **9 066** | films ayant reçu au moins un avis dans `ratings_small` |

Pour situer ces 9 066 films, il existe deux autres périmètres :

| | | |
|---|---|---|
| Films du jeu réduit | 9 125 | lignes de `links_small` : 59 films de plus, présents mais jamais notés |
| Films du catalogue | 45 436 | ids distincts de `movies_metadata` avant nettoyage (45 433 après, question 5) : tout TMDB, dont 36 000 films hors du jeu réduit |

Le chiffre qui compte pour la suite est **9 066** : c'est le nombre de colonnes de la matrice
utilisateurs × films avant séparation train/test (8 384 après, les films notés en train). Chaque utilisateur a noté au moins 20 films (médiane 71). La matrice n'est remplie
qu'à **1,6 %** : elle est très creuse, ce sera la difficulté principale du filtrage collaboratif. Un tiers des films
n'ont qu'un seul avis, et 52 % des notes sont ≥ 4 : les utilisateurs notent surtout ce qu'ils ont aimé.

### 4) Rôle de `links_small.csv`

> Quelle est l'utilité du fichier `links_small.csv` par rapport aux deux autres fichiers ?

In [ ]:
# Le film movieId = 1 dans ratings : quel est-il ?
tmdb_id = int(links.loc[links["movieId"] == 1, "tmdbId"].iloc[0])
print(f"ratings movieId = 1  ->  links tmdbId = {tmdb_id}  ->  movies_metadata :")
display(movies.loc[movies["id"] == str(tmdb_id), ["id", "imdb_id", "title", "release_date"]])

print("movieId de ratings absents de links :", (~ratings["movieId"].isin(links["movieId"])).sum())
print("Films de links sans tmdbId          :", links["tmdbId"].isna().sum())
print("tmdbId de links absents de metadata :",
      (~links["tmdbId"].dropna().astype(int).astype(str).isin(movies["id"])).sum())

**Réponse.** Les avis utilisent des identifiants **MovieLens** (`movieId`), les métadonnées des
identifiants **TMDB** (`id`) : les deux tables n'ont aucune clé commune. `links_small` est la **table
de correspondance** qui permet de passer de l'un à l'autre, donc de retrouver le titre d'un film noté
(`movieId` 1 → `tmdbId` 862 → *Toy Story*).

Limite : 13 films n'ont pas de `tmdbId` et 30 `tmdbId` sont absents du catalogue. Ces films pourront
être notés et recommandés, mais pas affichés avec leur titre.

### 5) Nettoyage de `movies_metadata`

> Nettoyer la table `movies_metadata.csv` : convertir tous les ID de films au format numérique (en retirant les films dont les ID ne sont pas numériques), ne conserver qu'une ligne par ID unique.

Le nettoyage se fait en mémoire : le CSV n'est jamais modifié. On garde `movies` (brut) et on crée `movies_clean`.

In [ ]:
# 1. ID au format numérique : les valeurs non convertibles deviennent NaN
id_num = pd.to_numeric(movies["id"], errors="coerce")
print(f"Lignes dont l'id n'est pas numérique : {id_num.isna().sum()}")
display(movies.loc[id_num.isna(), ["adult", "budget", "id", "title", "release_date", "popularity", "revenue"]])

Ces trois lignes sont **décalées** : un morceau de synopsis dans `adult`, un chemin d'affiche dans
`budget`, une date dans `id`, pas de titre. Elles sont inexploitables.

In [ ]:
# 2. Retrait de ces lignes et conversion en entier
movies_clean = movies.loc[id_num.notna()].copy()
movies_clean["id"] = id_num.loc[id_num.notna()].astype(int)

# 3. Doublons sur l'id
doublons = movies_clean[movies_clean.duplicated("id", keep=False)].sort_values("id")
print(f"Lignes en doublon sur l'id : {doublons['id'].duplicated().sum()} "
      f"({doublons['id'].nunique()} ids concernés, {doublons.duplicated().sum()} doublons strictement identiques)")

# Deux exemples : un doublon identique (105045) et un doublon qui diffère (4912)
display(doublons.loc[doublons["id"].isin([105045, 4912]),
                     ["id", "title", "release_date", "popularity", "vote_count", "revenue"]])

cols_diff = {c for _, g in doublons.groupby("id") for c in g.columns if g[c].astype(str).nunique() > 1}
print("Colonnes qui diffèrent entre doublons :", cols_diff)

In [ ]:
movies_clean = movies_clean.drop_duplicates("id", keep="first").reset_index(drop=True)

print(f"Avant : {len(movies):,} lignes  ->  après : {len(movies_clean):,} lignes")
print("id unique :", movies_clean["id"].is_unique, "| dtype :", movies_clean["id"].dtype)

**Réponse.** Deux opérations :

1. **3 lignes** ont un `id` non numérique (lignes décalées de l'export) : retirées, puis `id` converti en entier.
2. **30 lignes** sont en doublon (29 films, dont un exporté trois fois). 17 sont strictement identiques ; les autres ne
   diffèrent que par `popularity` et, pour deux films, `vote_count`, des compteurs qui bougent dans le temps : ce sont
   plusieurs exports du même film. On garde la première.

Résultat : **45 433 films**, un par `id` entier, joignable directement à `tmdbId` de `links_small`.

### 6) Nettoyage de `ratings_small`

> Nettoyer la table `ratings_small.csv` : retirer les avis manquants et les ID d'utilisateurs et de films non conformes, et convertir tous les ID d'utilisateurs et de films au format numérique.

Un avis est conforme si : la note est présente et sur l'échelle MovieLens (0,5 à 5 par pas de 0,5),
les identifiants sont des entiers strictement positifs, et le couple (utilisateur, film) est unique.

In [ ]:
n_avant = len(ratings)
ratings_clean = ratings.copy()

# 1. Avis manquants
manquants = ratings_clean[["userId", "movieId", "rating"]].isna().any(axis=1)
print(f"Avis avec valeur manquante          : {manquants.sum()}")
ratings_clean = ratings_clean[~manquants]

# 2. Identifiants conformes -> entiers
for col in ["userId", "movieId"]:
    num = pd.to_numeric(ratings_clean[col], errors="coerce")
    conforme = num.notna() & (num > 0) & (num % 1 == 0)
    print(f"{col:<8} non conformes               : {(~conforme).sum()}")
    ratings_clean = ratings_clean[conforme].assign(**{col: num[conforme].astype(int)})

# 3. Notes conformes
note_ok = ratings_clean["rating"].between(0.5, 5) & ((ratings_clean["rating"] * 2) % 1 == 0)
print(f"Notes hors échelle MovieLens        : {(~note_ok).sum()}")
ratings_clean = ratings_clean[note_ok]

# 4. Doublons (utilisateur, film)
dup = ratings_clean.duplicated(["userId", "movieId"])
print(f"Doublons (userId, movieId)          : {dup.sum()}")
ratings_clean = ratings_clean[~dup].reset_index(drop=True)

# Films sans métadonnées (conservés : le filtrage collaboratif n'en a pas besoin)
sans_meta = ~ratings_clean["movieId"].isin(links.loc[links["tmdbId"].isin(movies_clean["id"]), "movieId"])
print(f"Avis sur un film sans métadonnées   : {sans_meta.sum()} (conservés)")

print(f"\nAvant : {n_avant:,} avis  ->  après : {len(ratings_clean):,} avis")
print(ratings_clean.dtypes.to_string())

**Réponse.** `ratings_small` est déjà propre : aucun avis manquant, aucun identifiant non conforme,
aucun doublon, et les identifiants sont déjà des entiers. Le code reste défensif pour fonctionner sur
un fichier moins propre. Les **100 004 avis** sont conservés, y compris les 194 qui portent sur un film
sans métadonnées : le filtrage collaboratif n'utilise que les notes.

### 7) Les 10 films les plus anciens

> Déterminer les 10 films les plus anciens du dataset.

In [ ]:
# Conversion de release_date en date ; les valeurs invalides deviennent NaT
movies_clean["release_date"] = pd.to_datetime(movies_clean["release_date"], errors="coerce")

print("Films sans date de sortie :", movies_clean["release_date"].isna().sum())
futur = movies_clean[movies_clean["release_date"] > "2017-12-31"]
print(f"Films datés après 2017 (extraction du dataset) : {len(futur)}")
display(futur[["id", "title", "release_date", "status", "vote_count"]])

In [ ]:
cols = ["id", "title", "release_date", "runtime", "original_language", "vote_count"]
movies_clean.dropna(subset=["release_date"]).nsmallest(10, "release_date")[cols]

**Réponse.** Anomalies repérées avant de trier : 87 films sans date, et 6 films datés de 2018 ou
2020, donc pas encore sortis quand le dataset a été extrait (2017).

Les 10 films les plus anciens vont de **1874** (*Passage of Venus*) à **1890** (*Monkeyshines*) : des
films d'une minute des pionniers du cinéma, à l'époque des premières expériences de chronophotographie.

### 8) Les 10 plus gros succès au box-office

> Déterminer les 10 films ayant eu le plus gros succès au box-office (colonne `revenue`).

In [ ]:
movies_clean["revenue"] = pd.to_numeric(movies_clean["revenue"], errors="coerce")
rev = movies_clean["revenue"]

print(f"revenue manquant (NaN)  : {rev.isna().sum():>6,}")
print(f"revenue = 0             : {(rev == 0).sum():>6,}  ({(rev == 0).mean():.1%} des films)")
print(f"0 < revenue < 1 000     : {rev.between(1, 999).sum():>6,}")
print(f"revenue >= 1 000        : {(rev >= 1000).sum():>6,}")

# Exemples de valeurs suspectes : quelques dollars de recettes pour des films sortis en salle
display(movies_clean.loc[rev.between(1, 999), ["title", "release_date", "budget", "revenue"]].head(5))

In [ ]:
top_revenue = movies_clean.nlargest(10, "revenue").copy()
top_revenue["revenue (M$)"] = (top_revenue["revenue"] / 1e6).round(0)
top_revenue["budget (M$)"] = (pd.to_numeric(top_revenue["budget"], errors="coerce") / 1e6).round(0)
top_revenue[["id", "title", "release_date", "revenue (M$)", "budget (M$)", "vote_average", "vote_count"]]

**Réponse.** La colonne `revenue` est très incomplète : **84 % des films ont 0**, qui veut dire
"inconnu", et 154 films affichent quelques dollars (unité fausse). Cela ne gêne pas le classement des
plus gros succès, qui ne regarde que les grandes valeurs.

Le top 10 va d'**Avatar** (2,79 milliards de dollars) à **Beauty and the Beast** (1,26), en passant par
*Star Wars : The Force Awakens*, *Titanic*, *The Avengers*, *Jurassic World* et *Furious 7*. Les recettes
sont en dollars courants, non corrigées de l'inflation, d'où la domination des films récents.

### 9) Séparation entraînement / test

> Nous aurons besoin d'une séparation entraînement/test pour tester nos modèles. Que faut-il séparer en deux ? Effectuer une séparation entraînement/test de sorte à ce que chaque utilisateur en test ait une chance de recevoir de bonnes recommandations. Comment avez-vous fait ?

*Entraînement / test* : on cache une partie des avis (le **test**) et on construit le modèle sur le reste (le **train**).
On juge ensuite le modèle sur sa capacité à retrouver ce qu'on lui a caché, comme un examen dont on connaît les réponses.
*Stratifier* : tirer séparément dans chaque catégorie (ici films aimés / non aimés) pour que chaque catégorie soit
représentée. *Graine aléatoire* : nombre qui fixe le tirage, pour retomber sur le même découpage à chaque exécution.

In [ ]:
SEUIL_PERTINENT = 4.0   # un film noté >= 4 est considéré comme "aimé" (servira aussi pour les métriques top k)
FRAC_TEST = 0.2

# La fonction de séparation est dans donnees.py ; on affiche sa source pour la lecture
print(inspect.getsource(split_par_utilisateur))

train, test = split_par_utilisateur(ratings_clean, FRAC_TEST, SEUIL_PERTINENT, RANDOM_STATE)
print(f"train : {len(train):>6,} avis ({len(train) / len(ratings_clean):.1%})")
print(f"test  : {len(test):>6,} avis ({len(test) / len(ratings_clean):.1%})")

In [ ]:
# Vérifications : chaque utilisateur est-il bien "testable" ?
par_user_train = train.groupby("userId").size()
par_user_test = test.groupby("userId").size()
aimes_test = test[test["rating"] >= SEUIL_PERTINENT].groupby("userId").size()

print(f"Utilisateurs en train / en test          : {train['userId'].nunique()} / {test['userId'].nunique()}")
print(f"Avis par utilisateur en train (min/méd.) : {par_user_train.min()} / {par_user_train.median():.0f}")
print(f"Avis par utilisateur en test  (min/méd.) : {par_user_test.min()} / {par_user_test.median():.0f}")
print(f"Utilisateurs sans film aimé en test      : {test['userId'].nunique() - len(aimes_test)}")
aimes_train = train[train["rating"] >= SEUIL_PERTINENT].groupby("userId").size()
print(f"Utilisateurs sans film aimé en train     : {train['userId'].nunique() - len(aimes_train)}",
      sorted(set(train["userId"]) - set(aimes_train.index)))
part_test = par_user_test / (par_user_train + par_user_test)
print(f"Part de test par utilisateur (min/méd/max): {part_test.min():.1%} / {part_test.median():.1%} / {part_test.max():.1%}")
print(f"Note moyenne train / test                : {train['rating'].mean():.3f} / {test['rating'].mean():.3f}")
print(f"Films aimés en test par utilisateur (méd.): {aimes_test.median():.0f}")
print("Avis communs train/test                  :", len(train.merge(test, on=["userId", "movieId"])))

absents = ~test["movieId"].isin(train["movieId"])
print(f"Avis de test sur un film absent du train : {absents.sum()} ({absents.mean():.1%})")

In [ ]:
# Un exemple concret : l'utilisateur 221, 20 avis dont un seul film aimé
u = ratings_clean[ratings_clean["userId"] == 221].copy()
u["côté"] = np.where(u["movieId"].isin(test.loc[test["userId"] == 221, "movieId"]), "test", "train")
u["aimé"] = u["rating"] >= SEUIL_PERTINENT
print(u.groupby(["aimé", "côté"]).size().unstack(fill_value=0))
u.sort_values("rating", ascending=False)[["movieId", "rating", "aimé", "côté"]].head(6)

**Réponse.**

**On sépare les avis**, pas les utilisateurs. Pour recommander à quelqu'un, le modèle doit connaître
une partie de ses goûts : on lui cache donc 20 % des avis de chaque utilisateur, et on regarde s'il
les retrouve.

**Comment.** Le tirage est fait **utilisateur par utilisateur** (un tirage global pourrait laisser un
utilisateur avec presque rien en train), et **stratifié sur "film aimé (note ≥ 4) / non aimé"** avec au
moins un avis par strate : chaque utilisateur a ainsi au moins un film aimé en test, sans quoi aucune
recommandation ne pourrait être jugée bonne. Un seul générateur aléatoire, initialisé avec
`RANDOM_STATE`, sert à tous les utilisateurs : le tirage est reproductible et vraiment aléatoire.

**Résultat.** 79 976 avis en train, 20 028 en test. Les 671 utilisateurs sont des deux côtés, avec au
moins 15 avis en train et 4 en test, et tous ont au moins un film aimé en test. Aucun avis en commun.

**Exemple.** L'utilisateur 221 a 20 avis, un seul aimé : ce film part en test (au moins un par strate), avec 4 de
ses 19 films non aimés. Il garde 15 avis en train. Conséquence de cette règle : 3 utilisateurs (79, 221, 579), qui
n'ont qu'un seul film aimé dans tout le jeu, n'en ont aucun en train. Acceptable, ils gardent 43, 15 et 16 avis en train.

**Limite.** 745 avis de test (3,7 %) concernent des films que personne n'a notés en train : le filtrage
collaboratif ne pourra pas les prédire, on le gérera à l'évaluation. Une séparation temporelle (derniers
avis en test) serait plus réaliste mais n'est pas demandée ici.

## Exercice 2 – Un bon système de filtrage collaboratif

> Chargeons-nous dans un premier temps de concevoir un système de filtrage collaboratif *user-based*, simple mais efficace.

Principe : pour recommander à un utilisateur, on cherche ses **voisins** (les utilisateurs qui notent comme lui) et on lui
propose les films qu'ils ont aimés et qu'il n'a pas vus. Tout est construit à partir de `train` ; `test` ne sert qu'à mesurer.

### 1) Mesure de similarité et nombre de voisins

> Choisir une mesure de similarité et un nombre de voisins.

*Similarité* : un nombre qui dit à quel point deux utilisateurs notent pareil. Pour la calculer, chaque utilisateur est
vu comme un *vecteur* : la liste de ses notes, une case par film. Les *voisins* d'un utilisateur sont les k utilisateurs
qui lui sont le plus similaires ; k est un réglage à choisir.

In [ ]:
# Sur combien de films deux utilisateurs peuvent-ils être comparés ? (train uniquement)
presence = train.pivot(index="userId", columns="movieId", values="rating").notna().astype(int)
communs = presence.to_numpy() @ presence.to_numpy().T          # films notés en commun, par paire
paires = communs[np.triu_indices_from(communs, k=1)]           # chaque paire une fois

print(f"Films en commun par paire : médiane {np.median(paires):.0f}, "
      f"{(paires < 5).mean():.0%} des paires en ont moins de 5")
voisins_fiables = (communs >= 5).sum(axis=1) - 1
print(f"Utilisateurs avec >= 5 films en commun, par utilisateur : médiane {np.median(voisins_fiables):.0f}")

In [ ]:
K_VOISINS = 30   # similarité : cosinus (voir la réponse ci-dessous)

**Réponse.**

Le cours cite quatre mesures de similarité entre deux vecteurs de notes :

| Mesure | Principe | Ici |
|---|---|---|
| **Cosinus** | angle entre les deux vecteurs | **retenue** : c'est la mesure du cours ; insensible à la longueur du vecteur (gros ou petit noteur), seuls les films vus par les deux comptent dans le produit scalaire, calcul matriciel d'un coup. Limite : elle ne corrige pas le niveau d'un utilisateur sévère ou généreux, c'est le rôle du centrage (exercice 4) |
| Pearson | corrélation des notes sur les films en commun | proche du cosinus sur notes centrées, l'amélioration « centrage des notes », réservée à l'exercice 4 |
| Euclidienne | longueur du segment entre les deux points | pénalise les gros noteurs (vecteur long) |
| Jaccard | proportion de films vus en commun | ignore les notes |

**k = 30 voisins** pour commencer. Deux utilisateurs ont en médiane 4 films en commun, et 55 % des paires en ont moins
de 5 : une similarité repose souvent sur très peu de films, il faut plusieurs dizaines de voisins pour lisser ce bruit.
Trop de voisins, à l'inverse, noient la cible dans des utilisateurs qui ne lui ressemblent plus. Chaque utilisateur a en
médiane 294 autres utilisateurs avec au moins 5 films en commun, il y a donc de quoi choisir ; on vérifiera à la
question 3 que les 30 voisins retenus partagent bien assez de films avec l'utilisateur. Valeur à optimiser à l'exercice 4.

### 2) Matrice des avis et imputation

> Calculer une matrice des avis utilisateur/films. Celle-ci sera creuse (de très nombreuses valeurs manquantes). Comment l'imputer ?

*Matrice des avis* : un tableau avec une ligne par utilisateur et une colonne par film, la note dans la case.
Elle est *creuse* car presque toutes les cases sont vides (personne n'a vu tous les films). *Imputer* : décider quelle
valeur mettre dans les cases vides pour pouvoir faire les calculs. Le `pivot` de pandas construit ce tableau à partir
de la liste d'avis.

In [ ]:
# Matrice des avis (train) : une ligne par utilisateur, une colonne par film, NaN si non noté
R = train.pivot(index="userId", columns="movieId", values="rating")
print(f"{R.shape[0]} utilisateurs x {R.shape[1]} films, {R.notna().to_numpy().mean():.1%} de cases remplies, "
      f"{R.memory_usage().sum() / 1e6:.0f} Mo")

# Imputation : 0 pour les films non notés
R0 = R.fillna(0.0)
print("Cases vides restantes :", R0.isna().to_numpy().sum())

films_populaires = R.notna().sum().nlargest(8).index   # aperçu sur les 8 films les plus notés, avant / après
display(R[films_populaires].head(5))
R0[films_populaires].head(5)

**Réponse.**

La matrice `R` a **671 lignes × 8 384 colonnes** (les films notés au moins une fois en train ; les 682 autres films n'apparaissent que dans le test) et **1,4 %** de cases
remplies. En dense elle tient en mémoire (45 Mo), pas besoin de format creux. Pour calculer une similarité cosinus, il
faut une valeur dans chaque case : c'est l'imputation. Solutions possibles :

| Imputation | Principe | Ici |
|---|---|---|
| **0** | film non vu = 0 | **retenue** : c'est la solution du cours ; un film non vu par l'un des deux utilisateurs ne pèse pas dans le produit scalaire, seuls les films vus par les deux comptent |
| Moyenne de l'utilisateur | remplir sa ligne avec sa note moyenne | rend toutes les lignes pleines : deux utilisateurs deviennent similaires via des films qu'aucun n'a vus |
| Moyenne du film | remplir la colonne avec la note moyenne du film | même défaut, et favorise les films bien notés |
| Centrage puis 0 | retirer à chaque utilisateur sa moyenne, puis 0 = « ni aimé ni détesté » | proche de la corrélation de Pearson ; c'est l'amélioration de l'exercice 4 |
| Modèle | apprendre les cases vides (factorisation matricielle) | approche *model-based* du cours, hors périmètre user-based |

Limite du 0 : il ressemble à une très mauvaise note alors qu'il signifie « pas vu ». Deux utilisateurs qui ont vu les
mêmes films paraissent proches même s'ils les ont notés à l'opposé. L'exercice 4 mesurera ce que le centrage apporte.

Point important pour la suite : `R0` ne sert qu'à **calculer les similarités**. Pour prédire une note, on ne moyenne que
les voisins qui ont réellement noté le film (`R`, avec ses NaN), jamais les 0 imputés.

### 3) Le système de filtrage collaboratif user-based

> Concevoir un système de filtrage collaboratif user-based. L'entraîner sur la base d'entraînement et le tester sommairement :
> vérifier qu'il parvient à donner des avis pour un utilisateur donné, afficher les noms des films préférés d'un utilisateur
> au hasard et les films qu'il recommande, et vérifier la cohérence des recommandations.

*Moyenne pondérée* : une moyenne où chaque note compte proportionnellement à un poids, ici la similarité du voisin :
un voisin très proche pèse plus qu'un voisin lointain. *Hyperparamètre* : un réglage fixé à la main avant
l'entraînement (k, le seuil de voisins), par opposition à ce que le modèle calcule lui-même. Le système est écrit sous
forme de *classe* Python.

Algorithme du cours : similarité cosinus entre utilisateurs, k voisins, note prédite = moyenne des notes des voisins
pondérée par leur similarité, en ne comptant que les voisins qui ont vu le film. Si aucun ne l'a vu, on prédit la note
moyenne de l'utilisateur. Écart assumé avec l'exemple du cours, où un voisin qui n'a pas vu le film compte pour 0 dans la
moyenne : on l'exclut, sinon « pas vu » vaudrait la pire note possible.

Le modèle est écrit dans `modeles.py` (affiché ci-dessous) pour pouvoir changer ses réglages d'une ligne à l'exercice 4.

In [ ]:
# Le modèle est dans modeles.py ; on affiche sa source pour la lecture
print(inspect.getsource(RecoUserBased))

In [ ]:
reco = RecoUserBased(k=K_VOISINS).fit(train)
print(f"Similarités : {reco.sim.shape}, min {reco.sim.to_numpy().min():.2f}, max {reco.sim.to_numpy().max():.2f}")

In [ ]:
# Titre et année d'un movieId, via links (tmdbId) puis movies_clean (id)
titres = (links.dropna(subset=["tmdbId"]).astype({"tmdbId": int})
          .merge(movies_clean[["id", "title", "release_date"]], left_on="tmdbId", right_on="id")
          .set_index("movieId"))
titres["titre"] = titres["title"] + " (" + titres["release_date"].dt.year.astype("Int64").astype(str) + ")"
titres = titres["titre"]


def avec_titres(df):
    """Ajoute le titre en première colonne d'un DataFrame indexé par movieId."""
    return df.assign(titre=titres.reindex(df.index).fillna("(titre inconnu)")).set_index("titre", append=True)

In [ ]:
# Un utilisateur au hasard (générateur local : le tirage ne dépend pas des cellules exécutées avant)
user = int(np.random.default_rng(RANDOM_STATE).choice(train["userId"].unique()))
print(f"Utilisateur {user} : {reco.R.loc[user].notna().sum()} films notés en train, moyenne {reco.moyenne[user]:.2f}")
print("Voisins :", reco.voisins(user).round(2).to_dict())

# Les voisins retenus partagent-ils assez de films avec lui ? (pour tous les utilisateurs)
co_notes = np.concatenate([(reco.R.loc[reco.voisins(u).index].notna() & reco.R.loc[u].notna()).sum(axis=1).to_numpy()
                           for u in reco.R.index])
print(f"Films en commun entre un utilisateur et ses voisins retenus : médiane {np.median(co_notes):.0f}, "
      f"{(co_notes < 5).mean():.0%} des voisins en ont moins de 5")

pred = reco.predire(user)
print(f"Notes prédites : {len(pred)} films, de {pred.min():.2f} à {pred.max():.2f}")
pred.head()

In [ ]:
# Ses films préférés (train) et ses recommandations sans seuil, côte à côte
preferes = train[train["userId"] == user].set_index("movieId")["rating"].nlargest(10).to_frame("note")
print("Films préférés"); display(avec_titres(preferes))
print("Recommandations (min_voisins = 1)"); display(avec_titres(reco.recommander(user)))

Les dix recommandations sont des films vus par **un seul** voisin qui a mis 5 : une note prédite de 5 sur un seul avis
passe devant un film noté 4,6 par vingt voisins. On exige donc qu'un film ait été vu par plusieurs voisins pour être
recommandé. Sur cet utilisateur, un seuil de 2 laisse encore des films à deux avis, 5 ne garde que des classiques très
vus ; 3 est un compromis.

In [ ]:
# Le seuil ne joue que dans recommander : on compare 2, 3 et 5 sans ré-entraîner
for m in (2, 3, 5):
    reco.min_voisins = m
    r = reco.recommander(user).head(5)
    print(f"min_voisins = {m} :", " · ".join(f"{t} ({v} voisins)" for t, v in zip(titres.reindex(r.index).fillna("?"), r["voisins"])))

In [ ]:
MIN_VOISINS = 3   # voisins ayant vu un film pour pouvoir le recommander ; à optimiser à l'exercice 4

reco.min_voisins = MIN_VOISINS
print(f"Recommandations (min_voisins = {MIN_VOISINS})"); avec_titres(reco.recommander(user)).round(2)

**Réponse.**

Le système est la classe `RecoUserBased` de `modeles.py` : `fit` construit la matrice des avis et les 671 × 671
similarités cosinus (similarité maximale 0,74), `voisins` retient les k plus proches, `score` donne une valeur à
chacun des 8 384 films, `predire` la ramène sur l'échelle 0,5 à 5, `recommander` classe les films non vus.
L'entraînement et une prédiction prennent moins d'une seconde. Les 30 voisins retenus partagent en médiane 16 films
avec l'utilisateur, et seuls 11 % en partagent moins de 5 : le choix de k = 30 tient.

Sur l'utilisateur tiré au hasard (60), les notes prédites couvrent bien tous les films, de 0,5 à 5. Ses films
préférés sont des classiques et des films d'auteur (*The Godfather* I et II, *Blade Runner*, *Ed Wood*, *Gattaca*,
*The Pianist*, *Eternal Sunshine of the Spotless Mind*).

**Cohérence.** Sans seuil, les recommandations sont incohérentes : dix films confidentiels vus par un seul voisin.
Avec `MIN_VOISINS = 3`, elles deviennent crédibles pour ce profil : *Pulp Fiction*, *The Good, the Bad and the Ugly*,
*Annie Hall*, *Chinatown*, *Eraserhead*, *The Lives of Others*. Aucun blockbuster ni film pour enfants, et *Pulp Fiction*,
vu par 21 voisins, est une valeur sûre. La cohérence se juge ici à la lecture des titres ; des mesures
chiffrées (popularité, diversité) viendront à l'exercice 5. Ce seuil s'ajoute à
l'algorithme du cours ; il a été fixé à l'œil sur cet utilisateur (2 laisse des films à deux avis, 5 ne garde que des classiques très
vus) et c'est un hyperparamètre de plus pour l'exercice 4.

### 4) Absence de fuite de données

> S'assurer qu'aucune fuite de données n'est présente dans votre code.

Il y aurait fuite si une information de `test` servait à construire le modèle : un avis caché présent dans la matrice,
un film connu seulement par le test, ou un choix d'hyperparamètre fait sur le test. Le modèle ne reçoit que `train` ;
on le vérifie sur les objets construits.

In [ ]:
# 1. La matrice du modèle ne contient que les avis de train
print("Avis dans la matrice   :", reco.R.notna().to_numpy().sum(), "| avis en train :", len(train))
print("Films dans la matrice  :", reco.R.shape[1], "| films en train :", train["movieId"].nunique())

# 2. Aucun avis de test n'est dans la matrice : les cases (utilisateur, film) du test sont vides
test_connu = test[test["movieId"].isin(reco.R.columns)]
lig, col = reco.R.index.get_indexer(test_connu["userId"]), reco.R.columns.get_indexer(test_connu["movieId"])
assert (lig >= 0).all() and (col >= 0).all()          # -1 signalerait un utilisateur ou un film absent
print("Cases de test remplies :", np.isfinite(reco.R.to_numpy()[lig, col]).sum(), "sur", len(test_connu))

# 3. Les films connus seulement par le test sont inconnus du modèle
seulement_test = set(test["movieId"]) - set(train["movieId"])
print("Films seulement en test:", len(seulement_test), "| dans la matrice :", len(seulement_test & set(reco.R.columns)))

# 4. Le contrôle est sensible : un modèle entraîné sur train + test aurait d'autres similarités
sim_fuite = RecoUserBased(k=K_VOISINS).fit(pd.concat([train, test])).sim
print("Similarités identiques à un modèle entraîné avec le test :", np.allclose(sim_fuite, reco.sim))

# 5. La stratification du split sur la note change-t-elle le résultat ? Même modèle sur un tirage non stratifié
rng_s = np.random.default_rng(RANDOM_STATE)
test_ns = ratings_clean.groupby("userId").sample(frac=FRAC_TEST, random_state=rng_s)
train_ns = ratings_clean.drop(test_ns.index)
rmse_ns = rmse_mae(test_ns["rating"], predire_test(RecoUserBased(k=K_VOISINS).fit(train_ns), test_ns))[0]
print(f"RMSE du modèle : split stratifié {rmse_mae(test['rating'], predire_test(reco, test))[0]:.3f}, non stratifié {rmse_ns:.3f}")

**Réponse.** Aucune fuite :

- la matrice des avis contient exactement les 79 976 avis de train, sur les 8 384 films de train ;
- les 19 283 cases (utilisateur, film) du test qui existent dans la matrice sont toutes vides ;
- les 682 films connus seulement par le test sont absents de la matrice ;
- le contrôle est sensible : un modèle entraîné avec le test aurait d'autres similarités.
- la stratification du découpage ne change presque rien : RMSE 1,011 sur le split stratifié, 1,014 sur un tirage simple.

Dans le code, `test` n'apparaît qu'à la question 9 (création), dans les vérifications et dans les mesures finales.
Les réglages faits jusqu'ici (`K_VOISINS`, `MIN_VOISINS`) ont été choisis sans regarder le test : `MIN_VOISINS` a été
fixé en lisant des recommandations sur train. La séparation stratifie sur la note (film aimé ou non), ce qui garantit
un film aimé en test par utilisateur ; ce n'est pas une fuite, le modèle ne voit pas ces avis, et l'effet sur la RMSE
est mesuré ci-dessus : négligeable. Règle pour l'exercice 4 : les hyperparamètres seront optimisés sur un sous-ensemble de train mis de
côté (validation), pas sur le test, avec la même fonction `split_par_utilisateur`.

### 5) RMSE et MAE sur l'ensemble de test

> Calculer sur l'ensemble de test la RMSE et la MAE.

Pour chaque avis caché, le modèle prédit une note qu'on compare à la vraie. L'*erreur* est la différence entre les deux.

*MAE* (erreur absolue moyenne) : la moyenne des erreurs sans leur signe. Une MAE de 0,8 veut dire qu'en moyenne la note
prédite est à 0,8 point de la vraie. *RMSE* (racine de l'erreur quadratique moyenne) : on met les erreurs au carré
avant de moyenner, puis on prend la racine. Se lit comme la MAE mais compte davantage les grosses erreurs : se tromper de
2 points une fois coûte plus que se tromper de 1 point deux fois. Plus les deux sont petites, mieux c'est, sur une
échelle de notes de 0,5 à 5.

*Comment lire ces chiffres.* Un chiffre seul ne dit rien : une RMSE de 1,0 est bonne ou mauvaise selon ce qu'un
modèle trivial obtient. On compare donc à des *repères*, des « modèles » qui prédisent toujours une moyenne (globale,
de l'utilisateur, du film) ; un vrai modèle doit faire mieux qu'eux. Trois règles de lecture : la RMSE est toujours
supérieure ou égale à la MAE, et l'écart entre les deux grandit avec le nombre de grosses erreurs ; un modèle très simple qui
ne fait qu'apprendre le niveau de chaque utilisateur et de chaque film (repère « biais ») donne l'ordre de grandeur
de ce qu'on peut atteindre.

Les deux fonctions, `predire_test` et `rmse_mae`, sont dans `evaluation.py`.

In [ ]:
# Prédiction de chaque avis caché, puis RMSE et MAE (fonctions dans evaluation.py)
test_eval = test.copy()
test_eval["pred"] = predire_test(reco, test_eval)
rmse, mae = rmse_mae(test_eval["rating"], test_eval["pred"])
print(f"User-based (k = {K_VOISINS}) : RMSE = {rmse:.3f}   MAE = {mae:.3f}")

In [ ]:
# Repères : que vaudrait un modèle sans voisins ?
moy_globale = train["rating"].mean()
moy_user = train.groupby("userId")["rating"].mean()
moy_film = train.groupby("movieId")["rating"].mean()

# Repère « biais » : moyenne globale + écart moyen du film + écart moyen de l'utilisateur (calculé sur le reste)
biais_film = (train["rating"] - moy_globale).groupby(train["movieId"]).mean()
biais_user = (train["rating"] - moy_globale - biais_film.reindex(train["movieId"]).to_numpy()).groupby(train["userId"]).mean()
pred_biais = (moy_globale + biais_film.reindex(test_eval["movieId"]).fillna(0).to_numpy()
              + biais_user.reindex(test_eval["userId"]).fillna(0).to_numpy())

reperes = {
    "moyenne globale": np.full(len(test_eval), moy_globale),
    "moyenne de l'utilisateur": moy_user.reindex(test_eval["userId"]).to_numpy(),
    "moyenne du film": moy_film.reindex(test_eval["movieId"]).fillna(moy_globale).to_numpy(),
    "biais utilisateur + film": np.clip(pred_biais, 0.5, 5),
    f"user-based (k = {K_VOISINS})": test_eval["pred"].to_numpy(),
}
pd.DataFrame([(nom, *rmse_mae(test_eval["rating"], p)) for nom, p in reperes.items()],
             columns=["modèle", "RMSE", "MAE"]).set_index("modèle").round(3)

In [ ]:
# Où le modèle se trompe : films inconnus (repli) et erreurs selon la vraie note
inconnu = ~test_eval["movieId"].isin(reco.R.columns)
print(f"Avis sur film inconnu (repli sur la moyenne) : {inconnu.sum()}  RMSE = {rmse_mae(test_eval.loc[inconnu, 'rating'], test_eval.loc[inconnu, 'pred'])[0]:.3f}")
print(f"Avis sur film connu                          : {(~inconnu).sum()}  RMSE = {rmse_mae(test_eval.loc[~inconnu, 'rating'], test_eval.loc[~inconnu, 'pred'])[0]:.3f}")
test_eval["erreur"] = test_eval["pred"] - test_eval["rating"]
test_eval.groupby("rating")["erreur"].agg(["mean", "count"]).round(2).T

In [ ]:
# Deux causes mesurées : les prédictions sont tassées, et le niveau de l'utilisateur est ignoré
print(f"Écart-type des notes prédites / vraies : {test_eval['pred'].std():.2f} / {test_eval['rating'].std():.2f}")
niveau = moy_user.reindex(test_eval["userId"]).to_numpy()
print(f"Corrélation entre l'erreur (prédite - vraie) et le niveau de l'utilisateur : "
      f"{np.corrcoef(test_eval['erreur'], niveau)[0, 1]:.2f}")

In [ ]:
# RMSE selon le nombre de voisins (parmi les k) qui ont vu le film
n_vois_avis = pd.Series(0, index=test_eval.index)
for u, avis in test_eval.groupby("userId"):
    vus = reco.R.loc[reco.voisins(u).index].notna().sum()             # voisins ayant vu chaque film
    n_vois_avis[avis.index] = vus.reindex(avis["movieId"]).fillna(0).to_numpy()
tranche = pd.cut(n_vois_avis, [-1, 0, 1, 2, 4, 9, 30], labels=["0", "1", "2", "3-4", "5-9", "10+"])
par_tranche = test_eval.groupby(tranche, observed=True)["erreur"]
pd.DataFrame({"RMSE": par_tranche.apply(lambda e: np.sqrt((e ** 2).mean())), "avis": par_tranche.size()}).round(3).T

**Réponse.**

| Modèle | RMSE | MAE |
|---|---|---|
| Moyenne globale | 1,052 | 0,844 |
| Moyenne de l'utilisateur | 0,954 | 0,743 |
| Moyenne du film | 0,991 | 0,767 |
| Biais utilisateur + film | **0,908** | **0,694** |
| **User-based, k = 30** | 1,011 | 0,778 |

Le système obtient **RMSE = 1,011 et MAE = 0,778** : en moyenne, la note prédite est à 0,8 point de la vraie. C'est
mieux que la moyenne globale, mais **moins bien que la simple moyenne de l'utilisateur** (0,954), et loin du repère
« biais » (0,908), qui ne fait pourtant qu'ajouter le niveau du film à celui de l'utilisateur. `MIN_VOISINS` n'intervient pas ici : il ne sert qu'au
classement des recommandations, pas à la note prédite. Trois causes, mesurées dans
les cellules ci-dessus :

1. **Le niveau de l'utilisateur est ignoré.** La prédiction moyenne les notes brutes des voisins : un utilisateur sévère
   reçoit les notes de voisins plus généreux. L'erreur est corrélée au niveau de l'utilisateur (−0,29 : plus
   l'utilisateur note bas, plus on le surestime). Une moyenne pondérée reste toujours entre la plus basse et la plus
   haute note des voisins : par construction, elle ne peut pas suivre un utilisateur qui note systématiquement haut ou bas.
   C'est un résultat connu (Herlocker et al., 1999) : sans centrage, le user-based fait moins bien que la moyenne du film.
2. **Les prédictions sont tassées vers le milieu.** Leur écart-type (la dispersion autour de la moyenne) est de 0,67
   contre 1,05 pour les vraies notes : erreur moyenne de +2,6 sur les notes 0,5 et de −1,1 sur les notes 5. Une
   moyenne pondérée de plusieurs voisins lisse les extrêmes.
3. **Beaucoup de films sont peu vus par les voisins.** Quand au moins 10 voisins ont vu le film, la RMSE tombe à 0,87 ;
   avec un seul voisin elle monte à 1,30. Pour 2 138 avis (10 %), aucun des 30 voisins n'a vu le film et on se replie
   sur la moyenne de l'utilisateur, dont 745 sur des films absents de train.

Le centrage des notes par utilisateur, prévu à l'exercice 4, s'attaque directement aux
deux premières causes. Et la RMSE mesure la prédiction de notes, pas la qualité d'un top 10 : c'est l'objet de
l'exercice 3.

## Exercice 3 – Évaluation

> Afin de comparer nos modèles entre eux, il vaut mieux posséder des métriques qui ont du sens vis-à-vis de notre application.

### 1) Précision, rappel et NDCG top k

> Rechercher rapidement ce que signifient les termes "Précision top k", "Rappel top k" et "NDCG top k". Pourquoi ces
> métriques sont plus intéressantes pour notre cas pratique ?

**Le vocabulaire.** *Top k* : le système classe tous les films non vus par l'utilisateur et on ne garde que les k
premiers, ici k = 10, la liste qu'on lui montrerait. *Pertinent* : un film de cette liste est pertinent si l'utilisateur
l'a réellement aimé, c'est-à-dire s'il est dans ses avis cachés (le test) avec une note ≥ 4, le seuil de la question 9.
Un film absent du test n'est pas pertinent : on ne sait pas s'il l'aurait aimé, on ne peut pas le compter. Les trois
métriques se calculent **pour un utilisateur**, puis on prend la **moyenne sur les 671 utilisateurs**.

**Précision top k : « sur ce que j'ai montré, combien était bon ? »**

$$\text{précision@k} = \frac{\text{nombre de films pertinents dans les } k \text{ proposés}}{k}$$

Elle va de 0 (aucun bon film dans la liste) à 1 (que des bons films). Une précision de 0,2 se lit « 2 films sur 10
étaient des films que l'utilisateur a aimés ». C'est le point de vue de l'utilisateur qui regarde sa liste : plus elle
est haute, moins il y a de déchet. Ce qu'elle ne dit pas : si l'utilisateur avait 50 films aimés cachés et qu'on en
a trouvé 2, elle vaut 0,2 exactement comme s'il n'en avait que 2 et qu'on avait tout trouvé. Elle ignore aussi l'ordre
des 10 films.

**Rappel top k : « sur ce qu'il aimait, combien ai-je retrouvé ? »**

$$\text{rappel@k} = \frac{\text{nombre de films pertinents dans les } k \text{ proposés}}{\text{nombre de films pertinents de l'utilisateur}}$$

Même numérateur, autre dénominateur : le nombre total de films aimés cachés de l'utilisateur. Un rappel de 0,25 se lit
« on a retrouvé un quart de ses films aimés ». Il vaut 1 quand tous ses films aimés sont dans la liste, ce qui est
impossible s'il en a plus de k : avec 30 films aimés cachés et 10 places, le rappel plafonne à 10/30. C'est le point
de vue de la couverture : a-t-on manqué des choses ? Précision et rappel se complètent : proposer 1 000 films donnerait
un rappel énorme et une précision minuscule, en proposer 1 seul l'inverse. À k fixé, on regarde les deux.

**NDCG top k : « les bons films sont-ils en haut de la liste ? »** Les deux métriques précédentes traitent la liste
comme un sac : un bon film en position 10 vaut autant qu'en position 1. Or l'utilisateur lit de haut en bas et
s'arrête vite. Le NDCG se construit en quatre étapes :

1. **Gain** : chaque film pertinent rapporte 1, les autres 0.
2. **Décote** (*discounted*) : le gain est divisé par log₂(position + 1). En position 1 le film rapporte 1, en
   position 2 il rapporte 0,63, en position 3 0,50, en position 5 0,39, en position 10 0,29. Le logarithme fait
   décroître doucement : la position 1 vaut environ trois fois la position 10, pas dix fois.
3. **Cumul** (*cumulative*) : on additionne les gains décotés des k positions. C'est le DCG.
4. **Normalisation** (*normalized*) : on divise par le DCG du meilleur classement possible, celui qui mettrait tous
   les films pertinents de l'utilisateur en tête. Ce DCG idéal dépend du nombre de films pertinents qu'il a. Le
   résultat est entre 0 et 1, et vaut 1 si la liste est parfaite.

**Un exemple complet, sur l'utilisateur 60.** Ses avis cachés contiennent 8 films aimés : *Grave of the Fireflies*,
*Requiem for a Dream*, *The Big Lebowski* (5), *Psycho*, *Casino* (4,5), *Big*, *Desperado*, *Pee-wee's Big Adventure*
(4), et 3 films non aimés dont *Get Shorty* (3). Imaginons qu'un système lui propose cette liste :

| Position | Film proposé | Dans son test ? | Pertinent | Gain décoté |
|---|---|---|---|---|
| 1 | Psycho | oui, 4,5 | **oui** | 1 / log₂(2) = **1,00** |
| 2 | Pulp Fiction | non | non | 0 |
| 3 | Chinatown | non | non | 0 |
| 4 | The Big Lebowski | oui, 5 | **oui** | 1 / log₂(5) = **0,43** |
| 5 à 9 | Annie Hall, Eraserhead, … | non | non | 0 |
| 10 | Get Shorty | oui, mais 3 | non | 0 |

*Get Shorty* est dans le test mais noté 3 : il n'est pas pertinent, l'utilisateur ne l'a pas aimé. Les films absents du
test ne comptent pas non plus, même si certains sont peut-être d'excellentes idées : on n'a aucun moyen de le savoir.

| Métrique | Calcul | Valeur |
|---|---|---|
| Précision top 10 | 2 pertinents / 10 proposés | **0,20** |
| Rappel top 10 | 2 retrouvés / 8 aimés cachés | **0,25** |
| DCG | 1,00 + 0,43 | 1,43 |
| DCG idéal | ses 8 films aimés aux positions 1 à 8 : 1 + 0,63 + 0,50 + 0,43 + 0,39 + 0,36 + 0,33 + 0,32 | 3,95 |
| NDCG top 10 | 1,43 / 3,95 | **0,36** |

Si les deux mêmes films étaient en positions 7 et 10 au lieu de 1 et 4, précision et rappel ne bougeraient pas, mais le
DCG tomberait à 0,33 + 0,29 = 0,62 et le NDCG à 0,16. Seul le NDCG voit la différence entre une liste qui commence
bien et une liste qui finit bien.

**Pourquoi elles sont plus intéressantes que la RMSE ici.** L'utilisateur ne voit jamais une note prédite : il voit une
liste de 10 films. Prédire 3,48 au lieu de 3,5 pour *Magnolia* n'a aucune conséquence ; mettre ou non *Psycho* dans son
top 10 en a une. La RMSE juge toutes les cases de la matrice à égalité, y compris les films que l'utilisateur n'aurait
jamais regardés, alors que ces trois métriques ne jugent que le haut de la liste, là où se joue la recommandation. On
l'a vu à l'exercice 2 : la moyenne de l'utilisateur est imbattable en RMSE et incapable de classer quoi que ce soit,
puisqu'elle donne la même note à tous les films. Les trois métriques répondent enfin à des questions différentes, la
précision à la qualité de ce qu'on montre, le rappel à ce qu'on a manqué, le NDCG à l'ordre, et c'est bien ce qu'on
veut comparer entre deux systèmes.

**À quoi s'attendre.** On classe environ 8 000 films non vus pour en retenir 10, et chaque utilisateur n'a en médiane
que 8 films aimés cachés : des valeurs de quelques pourcents sont normales sur ce jeu de données. Le sujet prévient
qu'une valeur exactement nulle, elle, signale un bug.

### 2) Implémentation des métriques

> Implémenter des fonctions pour calculer ces métriques à partir d'un système de recommandation donné et d'un ensemble de test.

Trois fonctions dans `evaluation.py` : `pertinents_test` donne, pour chaque utilisateur, l'ensemble de ses films aimés
cachés ; `precision_rappel_ndcg` calcule les trois métriques d'un utilisateur à partir de sa liste de films proposés ;
`evaluer_top_k` fait le tour des utilisateurs du test et renvoie une ligne par utilisateur, qu'il reste à moyenner.
Le système est passé sous forme d'une fonction `recommander(user, k)` qui renvoie les films proposés dans l'ordre :
n'importe quel système, y compris un repère, s'évalue ainsi.

In [ ]:
# Les fonctions sont dans evaluation.py ; on affiche leur source pour la lecture
from evaluation import pertinents_test, precision_rappel_ndcg, evaluer_top_k
for f in (pertinents_test, precision_rappel_ndcg, evaluer_top_k):
    print(inspect.getsource(f))

In [ ]:
# Test sur l'exemple de la question 1 : 8 films aimés cachés, 2 retrouvés en positions 1 et 4
aimes = {"Psycho", "Big Lebowski", "Fireflies", "Requiem", "Casino", "Big", "Desperado", "Pee-wee"}
liste = ["Psycho", "Pulp Fiction", "Chinatown", "Big Lebowski", "Annie Hall",
         "Eraserhead", "Lives of Others", "Happiness", "Miller's Crossing", "Get Shorty"]
print("attendu 0.20 / 0.25 / 0.36  ->  obtenu", [round(x, 2) for x in precision_rappel_ndcg(liste, aimes)])

**Réponse.** Les trois fonctions redonnent les valeurs calculées à la main sur l'exemple de la question 1.
Un film proposé compte comme pertinent s'il est dans les avis cachés de l'utilisateur avec une note ≥ 4 ; les films
absents du test ne comptent pas. Le DCG idéal est calculé avec le nombre de films aimés cachés de l'utilisateur,
plafonné à k. Le gain du NDCG est binaire (1 si aimé, 0 sinon) plutôt que gradué par la note : c'est le choix le plus courant
et le plus lisible, et le seuil ≥ 4 est celui de toute l'évaluation. Tous les utilisateurs du test sont évalués, même
si le seuil ou le découpage changent : les moyennes restent comparables d'un modèle à l'autre.

### 3) Précision, rappel et NDCG top 10 du système

> Calculer la précision top 10, le rappel top 10 et le NDCG top 10 du système de recommandation que vous avez conçu.
> Si l'une de ces métriques vaut 0, c'est qu'il existe des erreurs dans votre code...

Comme pour la RMSE, deux repères pour lire les chiffres : proposer à chacun les 10 films **les plus notés** en train
qu'il n'a pas vus (le « top des ventes », sans aucune personnalisation), et proposer 10 films **au hasard**.

*Comment lire ces chiffres.* Ils sont entre 0 et 1, et ici petits, c'est normal : on choisit 10 films parmi 8 000, et
chaque utilisateur n'a en médiane que 8 films aimés cachés. Une précision de 0,05 se lit « en moyenne un demi-film
pertinent par liste de 10 » ; comme certains utilisateurs en ont deux ou trois, la part d'utilisateurs avec au
moins un bon film est plus faible que ne le suggère ce chiffre, on la compte à part. On les compare
aux repères, système par système : un NDCG plus élevé à précision égale veut dire que les bons films sont plus haut
dans la liste, ce qu'on vérifie directement avec leur position moyenne. Enfin, pour un utilisateur donné les trois valeurs sont souvent 0 ;
c'est la moyenne sur les 671 utilisateurs qui compte.

La taille de la liste est fixée par `TOP_K`. Ce n'est pas un réglage à optimiser : allonger la liste fait toujours
monter le rappel, sans que le système soit meilleur. C'est un choix d'affichage, qu'on fixe une fois pour comparer
les systèmes à taille égale.

In [ ]:
TOP_K = 10   # taille de la liste recommandée et évaluée

# Notre système : les TOP_K films recommandés à chaque utilisateur (calculés une fois), puis les trois métriques
listes = {u: list(reco.recommander(u, TOP_K).index) for u in test["userId"].unique()}
topk = evaluer_top_k(lambda u, k: listes[u][:k], test, SEUIL_PERTINENT, k=TOP_K)
print(f"Utilisateurs avec au moins un film pertinent dans leur top {TOP_K} : {(topk.iloc[:, 0] > 0).sum()} / {len(topk)}")
topk.mean().round(3)

In [ ]:
# Repères : les films les plus notés en train (non vus), et TOP_K films au hasard (non vus)
populaires = train.groupby("movieId").size().sort_values(ascending=False).index
vus_train = train.groupby("userId")["movieId"].apply(set)
rng_hasard = np.random.default_rng(RANDOM_STATE)
listes_pop = {u: [f for f in populaires if f not in vus_train[u]][:TOP_K] for u in listes}
listes_hasard = {u: list(rng_hasard.choice([f for f in populaires if f not in vus_train[u]], TOP_K, replace=False)) for u in listes}

comparaison = pd.DataFrame({
    "aléatoire": evaluer_top_k(lambda u, k: listes_hasard[u][:k], test, SEUIL_PERTINENT, k=TOP_K).mean(),
    "populaire": evaluer_top_k(lambda u, k: listes_pop[u][:k], test, SEUIL_PERTINENT, k=TOP_K).mean(),
    f"user-based (k = {K_VOISINS}, min_voisins = {MIN_VOISINS})": topk.mean(),
}).T.round(3)
comparaison

In [ ]:
# Où sont les bons films dans la liste, et quel rappel est atteignable ?
pertinents = pertinents_test(test, SEUIL_PERTINENT)
def positions_hits(listes):
    return [pos + 1 for u, films in pertinents.items() for pos, f in enumerate(listes[u]) if f in films]
pos_ub, pos_pop = positions_hits(listes), positions_hits(listes_pop)
print(f"Position moyenne des bons films dans le top {TOP_K} : user-based {np.mean(pos_ub):.1f} ({len(pos_ub)} bons films), "
      f"populaire {np.mean(pos_pop):.1f} ({len(pos_pop)} bons films)")

n_pert = np.array([len(f) for f in pertinents.values()])
print(f"Rappel@{TOP_K} maximal atteignable ({TOP_K} places par liste) : {np.mean(np.minimum(n_pert, TOP_K) / n_pert):.3f}")
n_absents = sum(len(f - set(train["movieId"])) for f in pertinents.values())
print(f"Films aimés cachés absents de train (jamais recommandables) : {n_absents} sur {n_pert.sum()}")

In [ ]:
# Que recommande-t-on ? Popularité des films proposés, et recouvrement avec le « top des ventes »
n_avis = train.groupby("movieId").size()
def profil(listes):
    pop = np.median([n_avis.get(f, 0) for films in listes.values() for f in films])
    distincts = len({f for films in listes.values() for f in films})
    recouv = np.mean([len(set(films) & set(listes_pop[u])) for u, films in listes.items()])
    return pd.Series({"popularité médiane (avis en train)": pop, "films distincts recommandés": distincts,
                      f"films en commun avec le top populaire (sur {TOP_K})": round(recouv, 2)})
print(f"Popularité médiane d'un film de train : {n_avis.median():.0f} avis")
pd.DataFrame({"user-based": profil(listes), "populaire": profil(listes_pop)})

**Réponse.**

| Système | Précision@10 | Rappel@10 | NDCG@10 |
|---|---|---|---|
| Aléatoire | 0,002 | 0,003 | 0,003 |
| Populaire (films les plus notés) | **0,109** | **0,100** | **0,147** |
| **User-based, k = 30, min_voisins = 3** | 0,050 | 0,066 | 0,060 |

Notre système obtient **précision 0,050, rappel 0,066, NDCG 0,060** : en moyenne un demi-film pertinent par top 10,
et 245 utilisateurs sur 671 (plus d'un sur trois) ont au moins un film aimé caché dans leur liste. Aucune métrique n'est
nulle, le test du sujet est passé. Le rappel maximal atteignable avec 10 places est de 0,79 : notre 0,066 en représente 8 %, et 293 des 10 324 films aimés
cachés sont absents de train, donc jamais recommandables.
Trois lectures :

- **C'est nettement mieux que le hasard** (0,002) : le système capte bien quelque chose des goûts de chacun.
- **C'est deux fois moins bien que le simple « top des ventes »**, qui propose les mêmes 10 films à tout le monde.
  Le NDCG des deux systèmes va dans le même sens : les bons films du repère populaire sont plus haut dans la liste
  (position moyenne 4,8) que les nôtres (6,0).
- **Nos recommandations sont confidentielles.** Les films proposés ont une popularité médiane de 54 avis en train,
  contre 200 pour le repère populaire ; 810 films distincts sont recommandés contre 57, et seul un film sur dix
  est commun avec le top des ventes. C'est bien de la
  personnalisation, mais elle se paie : un film peu vu a peu de chances de figurer dans les 20 % d'avis cachés de
  l'utilisateur, alors qu'un film très noté y est mécaniquement plus souvent. La métrique elle-même favorise les
  films populaires.

**Pourquoi.** Le classement se fait sur la note prédite, et un film vu par exactement 3 voisins qui ont tous mis 5
obtient 5,0 : il passe devant *Pulp Fiction* à 4,61 sur 21 avis. Le levier est le seuil de voisins par film, et il
faudra le régler à l'exercice 4 sur une validation, en surveillant en même temps la diversité de ce qu'on propose :
monter le seuil rapproche mécaniquement les recommandations du top des ventes.

## Exercice 4 – Améliorations

> 1) Optimiser les hyperparamètres de votre système de filtrage collaboratif user-based. Vous présenterez vos résultats
> dans un tableau et sélectionnerez avec justification le système qui vous semble être le meilleur. Vous commenterez la
> stabilité (ou l'instabilité) de vos résultats en fonction des variations des hyperparamètres.

### 1) Optimisation des hyperparamètres

Notre système a deux hyperparamètres : `k`, le nombre de voisins, et `min_voisins`, le nombre de voisins ayant vu un
film pour pouvoir le recommander. Ils ont été fixés à 30 et 3 sans vraie mise à l'épreuve.

*Validation croisée* : pour choisir des réglages sans toucher au test (règle posée à l'exercice 2), on découpe `train`
en 5 parts. Chaque part sert une fois d'ensemble caché, le modèle étant entraîné sur les 4 autres, et on moyenne les
5 résultats. Une validation simple (une seule part mise de côté, `split_par_utilisateur(train, ...)`) irait cinq fois
plus vite, mais ne dirait pas si un écart entre deux réglages est réel. *Stabilité* : l'écart d'un réglage entre les 5
parts, à comparer aux écarts entre réglages ; un résultat est stable si un petit changement de réglage ou de découpage
ne le bouleverse pas.

Critère de choix : le **NDCG@10**, la métrique de la liste qu'on montre, avec la RMSE en contrôle. On suit aussi la
popularité des films proposés et leur nombre, pour l'exercice 5.

In [ ]:
# Les deux fonctions sont dans donnees.py et evaluation.py ; on affiche leur source pour la lecture
for f in (parts_validation_croisee, evaluer_modele):
    print(inspect.getsource(f))

In [ ]:
# Grille sur les deux hyperparamètres, en validation croisée à 5 parts dans train (le test n'est pas touché)
GRILLE_K = [10, 20, 30, 50, 100]
GRILLE_MIN_VOISINS = [3, 5, 10, 15]
N_PARTS = 5
FICHIER_GRILLE = Path("resultats/grille_validation_croisee.csv")   # ~10 min de calcul : le résultat est gardé sur disque

if FICHIER_GRILLE.exists():
    grille = pd.read_csv(FICHIER_GRILLE)
else:
    parts = parts_validation_croisee(train, N_PARTS, SEUIL_PERTINENT, RANDOM_STATE)
    lignes = []
    for k in GRILLE_K:
        for m in GRILLE_MIN_VOISINS:
            for i, (train_fit, valid) in enumerate(parts):
                mesures = evaluer_modele(RecoUserBased(k=k, min_voisins=m).fit(train_fit), valid, SEUIL_PERTINENT, TOP_K)
                lignes.append({"k": k, "min_voisins": m, "part": i, **mesures})
    grille = pd.DataFrame(lignes)
    FICHIER_GRILLE.parent.mkdir(exist_ok=True)
    grille.to_csv(FICHIER_GRILLE, index=False)
print(f"{len(grille)} lignes : {len(GRILLE_K)} valeurs de k x {len(GRILLE_MIN_VOISINS)} seuils x {N_PARTS} parts")

In [ ]:
# Moyenne sur les 5 parts pour chaque réglage, et écart-type du NDCG entre parts (stabilité)
par_reglage = grille.groupby(["k", "min_voisins"])
resume = par_reglage.mean(numeric_only=True).drop(columns="part")
resume["NDCG écart-type"] = par_reglage["NDCG@10"].std()
resume.round(3)

In [ ]:
# Lecture en deux tableaux : NDCG@10 et RMSE selon k (lignes) et min_voisins (colonnes)
print("NDCG@10 (moyenne des 5 parts)"); display(resume["NDCG@10"].unstack("min_voisins").round(3))
print("RMSE (moyenne des 5 parts)"); resume["RMSE"].unstack("min_voisins").round(3)

In [ ]:
# La même chose en courbes, avec l'écart entre parts en barre d'erreur
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for m in GRILLE_MIN_VOISINS:
    r = resume.xs(m, level="min_voisins")
    axes[0].errorbar(r.index, r["NDCG@10"], yerr=r["NDCG écart-type"], marker="o", capsize=3, label=f"min_voisins = {m}")
axes[1].plot(r.index, r["RMSE"], marker="o")                        # une seule courbe : la RMSE ne dépend pas de min_voisins
axes[0].set(title="NDCG@10 en validation croisée", xlabel="k (voisins)", xscale="log")
axes[1].set(title="RMSE en validation croisée (même courbe pour tous les min_voisins)", xlabel="k (voisins)", xscale="log")
axes[0].legend(); plt.tight_layout()

Le meilleur réglage de la grille est sur son bord (k = 50, min_voisins = 15), et le tableau montre que les deux
réglages vont ensemble : le seuil doit valoir environ un tiers du nombre de voisins. On prolonge donc cette crête
au-delà de la grille avant de choisir.

In [ ]:
# Prolongement de la crête « min_voisins ≈ k / 3 » au-delà de la grille, même validation croisée
CRETE = [(50, 20), (50, 25), (100, 25), (100, 35), (100, 50)]
FICHIER_CRETE = Path("resultats/crete_validation_croisee.csv")

if FICHIER_CRETE.exists():
    crete = pd.read_csv(FICHIER_CRETE)
else:
    parts = parts_validation_croisee(train, N_PARTS, SEUIL_PERTINENT, RANDOM_STATE)
    crete = pd.DataFrame([{"k": k, "min_voisins": m, "part": i,
                           **evaluer_modele(RecoUserBased(k=k, min_voisins=m).fit(train_fit), valid, SEUIL_PERTINENT, TOP_K)}
                          for k, m in CRETE for i, (train_fit, valid) in enumerate(parts)])
    crete.to_csv(FICHIER_CRETE, index=False)

par_reglage = pd.concat([grille, crete]).groupby(["k", "min_voisins"])
resume = par_reglage.mean(numeric_only=True).drop(columns="part")
resume["NDCG écart-type"] = par_reglage["NDCG@10"].std()
resume.loc[[(30, 10), (50, 15)] + CRETE].round(3)

In [ ]:
# Réglage retenu (meilleur NDCG moyen), puis mesure UNIQUE sur le test, comparée au système initial
meilleur_k, meilleur_min = resume["NDCG@10"].idxmax()
print(f"Réglage retenu : k = {meilleur_k}, min_voisins = {meilleur_min}")

reco_initial = RecoUserBased(k=K_VOISINS, min_voisins=MIN_VOISINS).fit(train)
reco_retenu = RecoUserBased(k=meilleur_k, min_voisins=meilleur_min).fit(train)
pd.DataFrame({
    f"initial (k = {K_VOISINS}, min_voisins = {MIN_VOISINS})": evaluer_modele(reco_initial, test, SEUIL_PERTINENT, TOP_K),
    f"retenu (k = {meilleur_k}, min_voisins = {meilleur_min})": evaluer_modele(reco_retenu, test, SEUIL_PERTINENT, TOP_K),
}).round(3)

**Réponse.**

**Tableau.** 20 réglages de la grille plus 5 sur la crête, chacun mesuré 5 fois en validation croisée ; le tableau
complet est ci-dessus, la lecture en deux tableaux NDCG et RMSE, et les courbes. Le dossier `resultats/` garde les
125 mesures pour ne pas relancer la dizaine de minutes de calcul.

**Ce que dit la grille.**

- **La RMSE ne dépend que de k** : `min_voisins` ne joue que dans le classement, jamais dans la note prédite. Elle
  baisse régulièrement avec k, de 1,064 (10 voisins) à 1,004 (100 voisins) : plus de voisins, plus de films couverts,
  moins de replis sur la moyenne. Résultat stable et monotone, mais toujours au-dessus du repère « biais » de
  l'exercice 2 (0,908) : le réglage n'y change rien, seul le centrage (question 2) peut le faire.
- **Le top 10 dépend du couple, pas de chaque réglage séparément.** Seuil trop haut pour k, les listes se vident :
  (10, 10) et (10, 15) valent 0 par construction (un film ne peut pas être vu par 15 voisins sur 10, ce n'est pas un
  bug), (100, 50) tombe à 0,056. Seuil trop bas, les listes se remplissent de films confidentiels : (100, 3) donne
  0,008. Entre les deux, une crête autour de `min_voisins` ≈ k / 3 où le NDCG plafonne à 0,15 : (30, 10) 0,150,
  (50, 15) 0,151, (100, 35) 0,148. Le rapport fixe la forme, mais le niveau monte avec k : à 10 ou 20 voisins, le
  meilleur seuil ne dépasse pas 0,126.
- **Stabilité.** Entre les 5 parts, l'écart-type du NDCG vaut 0,002 à 0,008 : deux réglages à moins de 0,01 l'un de
  l'autre ne sont pas départageables, donc les trois réglages de la crête sont équivalents. En revanche, s'écarter de
  la crête coûte cher : à k = 50, passer le seuil de 15 à 25 divise le NDCG par 1,6, et à k = 100 le seuil 3 fait
  tout perdre. Le système est stable le long de la crête, instable en dehors : les deux hyperparamètres doivent être
  réglés ensemble, pas l'un après l'autre.

**Choix : k = 50, min_voisins = 15.** Meilleur NDCG moyen, et au milieu du plateau. Les deux voisins de crête ont
chacun un argument : (100, 35) a la meilleure RMSE (1,004 contre 1,018) mais ne propose plus que 126 films distincts
contre 219 ; (30, 10) en propose 289 mais avec la RMSE la plus haute des trois (1,031). Le réglage retenu est le
compromis ; k = 50 est dans le haut de la fourchette habituelle de la littérature (20 à 50 voisins). Si la diversité
devenait le critère, (30, 10) serait le choix.

**Sur le test, une seule fois.** Le réglage retenu passe la précision@10 de 0,050 à 0,148, le rappel de 0,066 à 0,163
et le NDCG de 0,060 à 0,191, devant le repère populaire (0,109 / 0,100 / 0,147). La RMSE gagne un peu, 0,999 contre
1,011. Le prix est visible : la popularité médiane des films proposés passe de 54 à 159 avis et le nombre de films
distincts de 810 à 249. Il faut le dire honnêtement : exiger qu'un film ait été vu par 15 des 50 voisins, c'est
ne garder que les films très vus. Le seuil agit comme un filtre de popularité, et une bonne part du gain vient de là.
Le système est devenu bon en se rapprochant du top des ventes, sans s'y réduire (249 films distincts contre 57) ;
c'est la question 1 de l'exercice 5.

Les valeurs sur le test sont plus hautes qu'en validation (0,191 contre 0,151) pour deux raisons : le modèle final est
entraîné sur tout train, un quart de données de plus ; et le test cache plus de films aimés par utilisateur qu'une
part de validation (un cinquième de tous les avis, contre un cinquième de train), or la précision@10 est plafonnée
pour un utilisateur qui a moins de 10 films aimés cachés. La comparaison entre réglages, elle, est faite à conditions
égales.

### 2) Une amélioration : le centrage des notes

> 2) Implémenter l'une des modifications suivantes au choix et les comparer avec votre système : le filtrage des
> utilisateurs trop peu actifs dans la base d'entraînement avant d'entraîner, le centrage des notes par utilisateur
> avant le calcul de similarité, le passage à un système de filtrage collaboratif item-based.

On choisit le **centrage**, parce qu'il vise le point faible mesuré à l'exercice 2 : le système ignore le niveau de
chaque utilisateur, et sa RMSE est moins bonne que la simple moyenne de l'utilisateur. Les deux autres options : le
filtrage des inactifs est prévu dans la classe (`min_avis`) mais chaque utilisateur a au moins 16 avis en train, peu
d'effet à attendre ; l'item-based est un autre système, gardé pour un bonus.

*Centrage* : avant de calculer les similarités, on retire à chaque utilisateur sa note moyenne. Un 4 chez quelqu'un
qui met 4,5 partout devient −0,5 ; un 4 chez quelqu'un qui met 2,5 partout devient +1,5. Deux utilisateurs sont alors
similaires s'ils aiment et détestent les mêmes films, pas s'ils notent au même niveau. Pour prédire, on moyenne ces
écarts chez les voisins puis on rajoute la moyenne de l'utilisateur : il retrouve son niveau. Sur notes centrées, une
similarité peut devenir négative (goûts opposés) ; c'est là que le `clip` à 0 de la classe entre en jeu.

Dans la classe, c'est l'option `centrer=True` : `X` contient les notes moins la moyenne de la ligne, et `score` rajoute
la moyenne. On compare brut et centré sur le réglage retenu, d'abord en validation croisée (mêmes 5 parts qu'à la
question 1), puis une seule fois sur le test. Le seuil `min_voisins` ayant été réglé pour le système brut, on vérifie
aussi deux autres seuils pour le centré.


In [ ]:
# Brut et centré, même réglage (k = 50, min_voisins = 15) et mêmes 5 parts qu'à la question 1
parts = parts_validation_croisee(train, N_PARTS, SEUIL_PERTINENT, RANDOM_STATE)

def moyenne_validation(Modele, **reglages):
    mesures = [evaluer_modele(Modele(**reglages).fit(t), v, SEUIL_PERTINENT, TOP_K) for t, v in parts]
    return pd.concat(mesures, axis=1).mean(axis=1)

validation = pd.DataFrame({
    "brut": moyenne_validation(RecoUserBased, k=meilleur_k, min_voisins=meilleur_min),
    "centré": moyenne_validation(RecoUserBased, k=meilleur_k, min_voisins=meilleur_min, centrer=True),
    "centré, seuil 10": moyenne_validation(RecoUserBased, k=meilleur_k, min_voisins=10, centrer=True),   # le seuil a été réglé pour le brut
    "centré, seuil 20": moyenne_validation(RecoUserBased, k=meilleur_k, min_voisins=20, centrer=True),
})
validation.round(3)

In [ ]:
# Une seule fois sur le test : le système retenu (brut) et le même, centré
reco_centre = RecoUserBased(k=meilleur_k, min_voisins=meilleur_min, centrer=True).fit(train)
print(f"Paires d'utilisateurs de similarité > 0 : brut {(reco_retenu.sim > 0).mean().mean():.0%}, centré {(reco_centre.sim > 0).mean().mean():.0%}")
pd.DataFrame({"brut": evaluer_modele(reco_retenu, test, SEUIL_PERTINENT, TOP_K),
              "centré": evaluer_modele(reco_centre, test, SEUIL_PERTINENT, TOP_K)}).round(3)

**Réponse.**

| Réglage (50, 15) | RMSE valid. | NDCG@10 valid. | RMSE test | NDCG@10 test | Films distincts (test) |
|---|---|---|---|---|---|
| Brut (système retenu) | 1,018 | **0,151** | 0,999 | 0,191 | 249 |
| Centré | **0,957** | 0,128 | **0,932** | 0,191 | 229 |

**Ce que le centrage gagne : la prédiction de notes.** RMSE de 1,018 à 0,957 en validation, de 0,999 à 0,932 sur le
test, MAE de 0,769 à 0,710. Le système centré bat enfin la moyenne de l'utilisateur (0,954) ; il reste derrière le
repère biais (0,908), qui connaît aussi le niveau de chaque film. C'est le gain attendu : la cause n° 1 de l'exercice
2, le niveau de l'utilisateur ignoré, est corrigée. Le gain est identique pour les trois seuils testés, puisque le
seuil ne joue pas dans la note prédite.

**Ce qu'il perd : un peu de classement, et c'est moins net.** En validation, le NDCG passe de 0,151 à 0,128, un écart
bien supérieur au bruit entre parts (0,002 à 0,008), et aucun des deux autres seuils ne le rattrape (0,120 et 0,074).
Sur le test, les deux systèmes font 0,191 : le centré est un peu meilleur en précision (0,150 contre 0,148), un peu
moins bon en rappel (0,142 contre 0,163). Deux raisons de ne pas conclure sur ce seul chiffre : le test n'est mesuré
qu'une fois, et on ne choisit pas sur le test. Ce qui change sous le capot : avec les notes centrées, 54 % des paires
d'utilisateurs ont une similarité positive contre 82 % en brut, parce que les goûts opposés sont maintenant à 0. Les
voisins sont donc différents, choisis sur les goûts et non plus sur le niveau et le volume de notes.

**Décision.** Le critère posé à la question 1 est le NDCG@10 en validation, et il donne le système brut. On le garde
pour recommander, et c'est lui que l'exercice 5 analyse. Le centré est le meilleur des deux pour prédire une note ;
si l'application était d'afficher une note prédite plutôt qu'un top 10, ce serait lui. La leçon : une amélioration
peut aider une métrique et pas l'autre, il faut savoir laquelle compte pour l'usage.


### Bonus : un système item-based

> (Troisième option de la question 2) le passage à un système de filtrage collaboratif item-based.

*Item-based* : même idée que le user-based, mais on compare les **films** entre eux au lieu des utilisateurs. Deux
films sont similaires si les mêmes personnes les ont notés de la même façon : similarité cosinus entre les colonnes de
la matrice des avis, avec le même 0 dans les cases vides (cours, pages 21 et 22). Pour recommander, l'algorithme du
cours (page 24) : on prend les n films que l'utilisateur a préférés, pour chacun ses k films les plus proches qu'il
n'a pas vus, et on prédit la note de chaque candidat par la moyenne des notes de l'utilisateur sur ses favoris,
pondérée par la similarité ; on classe par note prédite. Pour prédire la note d'un film donné (RMSE), on prend les
k films notés par l'utilisateur les plus proches de celui-là, même moyenne pondérée. La différence entre n et k : n
est le nombre de points de départ (les goûts de l'utilisateur), k le nombre de films ramenés par chaque point.

Deux différences avec le user-based. La prédiction part des notes de l'utilisateur lui-même : son niveau est conservé
sans centrage. Et la matrice de similarité grandit avec le nombre de films, pas d'utilisateurs : on ne garde que les
films ayant au moins `min_avis` avis en train (sinon leur similarité repose sur une poignée de noteurs), soit 3 081
films à 5 avis contre 671 utilisateurs. Trois réglages : `k`, `n_favoris` et `min_avis`.

*Règle de classement.* Le cours classe les candidats par note prédite. Pour l'item-based, cette note est une moyenne :
un candidat relié à un seul favori noté 5 obtient 5, même si la similarité est faible. La règle usuelle du top N
item-based (Deshpande et Karypis, 2004) ne divise pas : le score est la somme des similarités × notes des favoris, et
un film relié à dix favoris passe devant un film relié à un seul. La classe a les deux (`classement="note"` ou
`"somme"`) ; on les compare en validation croisée avec quelques réglages, puis le meilleur une seule fois sur le test
contre les deux systèmes user-based.


In [ ]:
# La classe est dans modeles.py ; on affiche sa source pour la lecture
print(inspect.getsource(RecoItemBased))

In [ ]:
# Item-based en validation croisée (mêmes 5 parts) : les deux règles de classement, quelques réglages
REGLAGES_ITEM = {
    "note, min_avis 5, n 10, k 10":    dict(classement="note",  min_avis=5,  n_favoris=10, k=10),
    "note, min_avis 50, n 10, k 10":   dict(classement="note",  min_avis=50, n_favoris=10, k=10),
    "somme, min_avis 5, n 20, k 20":   dict(classement="somme", min_avis=5,  n_favoris=20, k=20),
    "somme, min_avis 5, n 50, k 30":   dict(classement="somme", min_avis=5,  n_favoris=50, k=30),
    "somme, min_avis 20, n 50, k 30":  dict(classement="somme", min_avis=20, n_favoris=50, k=30),
}
validation_item = pd.DataFrame({nom: moyenne_validation(RecoItemBased, **r) for nom, r in REGLAGES_ITEM.items()})
validation_item.round(3).T

In [ ]:
# Une seule fois sur le test : le réglage item-based au meilleur NDCG en validation, contre les deux user-based
nom_item = validation_item.loc["NDCG@10"].idxmax()
reco_item = RecoItemBased(**REGLAGES_ITEM[nom_item]).fit(train)
print(f"Item-based retenu : {nom_item} ({len(reco_item.films)} films avec une similarité)")
pd.DataFrame({"user-based brut": evaluer_modele(reco_retenu, test, SEUIL_PERTINENT, TOP_K),
              "user-based centré": evaluer_modele(reco_centre, test, SEUIL_PERTINENT, TOP_K),
              "item-based": evaluer_modele(reco_item, test, SEUIL_PERTINENT, TOP_K)}).round(3)

**Réponse (bonus).**

| | RMSE valid. | NDCG@10 valid. | Films distincts valid. | RMSE test | NDCG@10 test |
|---|---|---|---|---|---|
| User-based brut (50, 15) | 1,018 | 0,151 | 219 | 0,999 | 0,191 |
| User-based centré (50, 15) | 0,957 | 0,128 | 167 | 0,932 | 0,191 |
| Item-based, classement par note (cours), min_avis 5 | **0,908** | 0,059 | 1 161 | | |
| Item-based, classement par somme, min_avis 20, n 50, k 30 (retenu) | 0,928 | **0,153** | 402 | **0,915** | **0,249** |

**Le meilleur pour prédire une note.** Avec les films à 5 avis ou plus, l'item-based obtient la meilleure RMSE de
tout le TP en validation, 0,908, devant le centré (0,957) et le brut (1,018). Il part des notes de l'utilisateur
lui-même, son niveau est donc conservé sans rien faire, et les similarités entre films sont plus fiables que celles
entre utilisateurs : un film populaire est noté par des centaines de personnes, un utilisateur n'a que quelques
dizaines de films en commun avec un autre. Monter `min_avis` dégrade un peu la RMSE, parce que les films écartés
n'ont plus de prédiction et se replient sur la moyenne de l'utilisateur.

**Le classement dépend entièrement de la règle.** Avec la règle du cours, classer par note prédite, le NDCG plafonne
à 0,066 : un candidat relié à un seul favori noté 5 par une similarité faible obtient 5 et passe en tête, et ce sont
souvent des films peu vus (popularité médiane 70 avis). Avec la somme des similarités × notes, un film relié à dix
favoris passe devant un film relié à un seul : le NDCG monte à 0,144 – 0,153, au niveau du user-based (0,151, écart
entre parts 0,004 à 0,007), sans aucun seuil de voisins. Et à égalité de NDCG, l'item-based propose des films moins
populaires (105 avis en médiane contre 134) et deux à quatre fois plus de films distincts. Le réglage retenu, au
meilleur NDCG en validation : somme, `min_avis` 20, n = 50 favoris, k = 30.

**Sur le test, une seule fois.** L'item-based retenu passe devant : NDCG 0,249 contre 0,191, précision 0,181 contre
0,148, RMSE 0,915 contre 0,999, avec des films moins populaires (128 avis contre 159) et 410 films distincts contre
249. L'écart est plus grand qu'en validation parce que le modèle final a un quart de données de plus, donc 1 043
films à 20 avis ou plus au lieu d'environ 800 : l'item-based profite plus de la taille du catalogue que le user-based.

**Bilan.** Le user-based reste le système du TP, construit et réglé aux exercices 2 à 4, et c'est lui que l'exercice 5
analyse. Mais le bonus montre qu'à ce stade, le meilleur système disponible est l'item-based avec classement par
somme, sur toutes les métriques à la fois ; l'exercice 5 le garde en point de comparaison et en amélioration.


## Exercice 5 – Analyse critique

> Justifier chaque réponse qui s'y prête à l'aide d'arguments numériques.
> 1) Votre système favorise-t-il les films populaires ? Estimez-vous que c'est une bonne chose ?
> 2) Les recommandations faites pour un utilisateur sont-elles diverses ?
> 3) Les recommandations faites au global sont-elles diverses ?
> 4) Les recommandations sont-elles réellement personnalisées ?
> 5) Comment amélioreriez-vous votre système en production ?

Le système analysé est celui du TP : le user-based retenu à l'exercice 4 (k = 50, min_voisins = 15). On met à côté
l'item-based du bonus et le repère « populaire », pour situer chaque chiffre. Les mesures portent sur les top 10 des
671 utilisateurs, calculés une fois ci-dessous.


In [ ]:
# Les top 10 de chaque système pour tous les utilisateurs, calculés une fois
utilisateurs = sorted(test["userId"].unique())
listes_ub = {u: list(reco_retenu.recommander(u, TOP_K).index) for u in utilisateurs}
listes_ib = {u: list(reco_item.recommander(u, TOP_K).index) for u in utilisateurs}
systemes = {"user-based retenu": listes_ub, "item-based (bonus)": listes_ib, "populaire": listes_pop}
print({nom: f"{sum(len(l) for l in listes.values())} films proposés" for nom, listes in systemes.items()})

### 1) Favorise-t-il les films populaires ?

*Popularité* d'un film : son nombre d'avis en train. *Biais de popularité* (cours, page 26) : un système qui ne
propose que des films déjà très vus, au détriment de films peu connus mais pertinents. On compare la popularité des
films proposés à celle du catalogue, et on regarde quelle part des recommandations tombe dans le top 50 des films les
plus notés.


In [ ]:
# Popularité des films proposés, comparée au catalogue et au top des ventes
top50 = set(populaires[:50])
def popularite(listes):
    films = [f for l in listes.values() for f in l]
    return pd.Series({"popularité médiane (avis en train)": np.median([n_avis.get(f, 0) for f in films]),
                      "part des recommandations dans le top 50": np.mean([f in top50 for f in films]),
                      "part de films à moins de 20 avis": np.mean([n_avis.get(f, 0) < 20 for f in films])})
print(f"Catalogue : {len(n_avis)} films, popularité médiane {n_avis.median():.0f} avis, "
      f"{(n_avis < 20).mean():.0%} des films ont moins de 20 avis")
display(pd.DataFrame({nom: popularite(l) for nom, l in systemes.items()}).round(3))

# Répartition : nombre d'avis des films proposés (une entrée par recommandation) contre le catalogue
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.logspace(0, 3, 30)
ax.hist(n_avis, bins=bins, density=True, alpha=0.4, label="catalogue (films de train)")
for nom in ("user-based retenu", "item-based (bonus)"):
    ax.hist([n_avis.get(f, 0) for l in systemes[nom].values() for f in l], bins=bins, density=True, histtype="step", lw=2, label=nom)
ax.set(xscale="log", xlabel="nombre d'avis du film (échelle log)", ylabel="densité", title="Popularité des films proposés")
ax.legend(); plt.tight_layout()

**Réponse.** **Oui, nettement.** Le film médian du catalogue a 3 avis et 88 % des films en ont moins de 20 ; le film
médian proposé par notre système en a 159, et aucun film proposé n'a moins de 20 avis. 60 % des recommandations
tombent dans le top 50 des films les plus notés (52 % pour l'item-based, 100 % pour le top des ventes). La courbe le
montre : la masse du catalogue est à gauche, nos recommandations sont toutes à droite.

La cause est connue depuis l'exercice 4 : exiger qu'un film ait été vu par 15 des 50 voisins ne laisse passer que
des films très vus. C'est le *biais de popularité* du cours (page 26) : les films peu populaires ne sont jamais
recommandés, même s'ils sont pertinents.

**Est-ce une bonne chose ?** En partie. Pour les métriques, oui : c'est ce seuil qui a fait passer le NDCG de 0,060
à 0,191, et un film populaire a plus de chances d'être dans les avis cachés. Pour l'utilisateur, c'est une valeur sûre
mais sans découverte : proposer *The Shawshank Redemption* à quelqu'un qui aime les classiques est rarement faux et
rarement utile, il le connaît probablement déjà. Pour la plateforme, c'est un défaut : 97 % du catalogue n'est jamais
proposé (question 3), donc jamais valorisé. Un système de production devrait tempérer ce biais, et l'item-based
montre que c'est possible sans perdre en qualité : films moins populaires (128 avis en médiane) et meilleur NDCG.


### 2) Les recommandations d'un utilisateur sont-elles diverses ?

*Diversité d'une liste* : est-ce que les dix films proposés se ressemblent tous, ou couvrent des choses différentes ?
On la mesure avec les genres des films (`movies_metadata`) : le nombre de genres distincts couverts par un top 10, et
la part des paires de films de la liste qui n'ont aucun genre en commun (1 = tous différents, 0 = tous pareils). Pour
situer, on mesure la même chose sur les dix films préférés de chaque utilisateur : ses propres goûts sont-ils variés ?


In [ ]:
# Genres de chaque film (movies_metadata via links), puis diversité de genres d'une liste
import ast
genres_tmdb = movies_clean.set_index("id")["genres"].map(lambda s: {g["name"] for g in ast.literal_eval(s)} if isinstance(s, str) else set())
genres_film = links.dropna(subset=["tmdbId"]).astype({"tmdbId": int}).set_index("movieId")["tmdbId"].map(genres_tmdb)

def diversite(films):
    """Genres distincts couverts par la liste, et part des paires de films sans genre commun."""
    g = [genres_film.get(f) if isinstance(genres_film.get(f), set) else set() for f in films]
    paires = [(a, b) for i, a in enumerate(g) for b in g[i + 1:]]
    return len(set().union(*g)), np.mean([not (a & b) for a, b in paires]) if paires else np.nan

preferes = {u: list(train[train["userId"] == u].nlargest(TOP_K, "rating")["movieId"]) for u in utilisateurs}
def diversite_moyenne(listes):
    d = np.array([diversite(l) for l in listes.values() if l])
    return pd.Series({"genres distincts par liste": d[:, 0].mean(), "paires sans genre commun": np.nanmean(d[:, 1])})
display(pd.DataFrame({**{nom: diversite_moyenne(l) for nom, l in systemes.items()},
                      "films préférés de l'utilisateur": diversite_moyenne(preferes)}).round(2))

# Exemple : l'utilisateur 60, ses préférés et son top 10 user-based, avec les genres
u = 60
def avec_genres(films):
    return pd.DataFrame({"titre": titres.reindex(films).fillna("(titre inconnu)").to_numpy(),
                         "genres": [", ".join(sorted(genres_film.get(f) or set())) for f in films]}, index=films)
print("Films préférés"); display(avec_genres(preferes[u]))
print("Top 10 user-based"); display(avec_genres(listes_ub[u]))

**Réponse.** **Diverses en genres, mais pas plus que les goûts de l'utilisateur.** Un top 10 user-based couvre en
moyenne 10,4 genres distincts et 43 % des paires de films de la liste n'ont aucun genre en commun. C'est presque
exactement le profil des dix films préférés de l'utilisateur (10,3 genres, 39 %) : le système reproduit la variété de
ses goûts, un peu plus. L'item-based fait pareil (11,0 et 44 %), et le top des ventes est le plus varié (11,5 et 54 %),
parce qu'il mélange sans tenir compte de personne.

L'exemple de l'utilisateur 60 le montre : ses préférés vont du *Parrain* à *Castle in the Sky* ; son top 10 est
cohérent, mais concentré sur le polar et le drame (huit films sur dix ont Crime ou Thriller) : les classiques du
genre que ses voisins ont aimés. Diversité honnête à l'échelle des genres, faible à l'échelle des thèmes.


### 3) Les recommandations au global sont-elles diverses ?

*Diversité globale* (ou couverture) : sur les 671 listes, combien de films différents sont proposés, quelle part du
catalogue cela représente, et à quel point les recommandations se concentrent sur quelques titres. Un système peut
faire des listes variées pour chacun tout en proposant les mêmes cinquante films à tout le monde.


In [ ]:
# Couverture du catalogue et concentration des recommandations
def concentration(listes):
    comptes = pd.Series([f for l in listes.values() for f in l]).value_counts()
    cumul = comptes.cumsum() / comptes.sum()
    return pd.Series({"films distincts proposés": len(comptes),
                      "part du catalogue couverte": len(comptes) / len(n_avis),
                      "films qui font la moitié des recommandations": int((cumul < 0.5).sum() + 1),
                      "part prise par les 10 films les plus proposés": comptes.iloc[:10].sum() / comptes.sum()})
display(pd.DataFrame({nom: concentration(l) for nom, l in systemes.items()}).round(3))

# Les films les plus souvent proposés par le user-based, et à combien d'utilisateurs
plus_proposes = pd.Series([f for l in listes_ub.values() for f in l]).value_counts().head(5)
avec_titres(plus_proposes.to_frame("utilisateurs")).assign(avis_en_train=lambda d: n_avis.reindex(d.index.get_level_values(0)).to_numpy())

**Réponse.** **Non.** Sur les 671 listes, le user-based ne propose que 249 films distincts, 3 % du catalogue, et
19 films suffisent à faire la moitié de toutes les recommandations ; les 10 films les plus proposés en représentent
35 %. *The Shawshank Redemption* est proposé à 331 utilisateurs sur 671, *The Godfather* à 317, *Schindler's List* à
295. C'est mieux que le top des ventes (57 films, 8 pour la moitié) et moins bien que l'item-based (410 films, 5 % du
catalogue, 32 pour la moitié). Autrement dit, chaque liste est variée, mais toutes puisent dans le même petit
réservoir de classiques : la conséquence directe du seuil de voisins.


### 4) Les recommandations sont-elles réellement personnalisées ?

*Personnalisation* : deux utilisateurs aux goûts différents reçoivent-ils des listes différentes ? On mesure le
nombre de films en commun entre les listes de deux utilisateurs, en moyenne sur toutes les paires, et le nombre de
films en commun avec le top des ventes. Un système non personnalisé donnerait la même liste à tout le monde : 10 films
en commun partout.


In [ ]:
# Films en commun entre deux listes (toutes les paires d'utilisateurs), et avec le top des ventes
def recouvrement(listes):
    ens = [set(l) for l in listes.values()]
    paires = [len(a & b) for i, a in enumerate(ens) for b in ens[i + 1:]]
    avec_pop = [len(set(l) & set(listes_pop[u])) for u, l in listes.items()]
    return pd.Series({"films en commun entre deux utilisateurs (moyenne)": np.mean(paires),
                      "paires d'utilisateurs sans aucun film en commun": np.mean(np.array(paires) == 0),
                      "films en commun avec le top des ventes": np.mean(avec_pop)})
display(pd.DataFrame({nom: recouvrement(l) for nom, l in systemes.items()}).round(2))

# Deux utilisateurs aux goûts opposés : celui qui note le plus haut et celui qui note le plus bas les films d'action
action = genres_film[genres_film.map(lambda g: isinstance(g, set) and "Action" in g)].index
note_action = train[train["movieId"].isin(action)].groupby("userId")["rating"].agg(["mean", "count"]).query("count >= 20")["mean"]
u_pour, u_contre = note_action.idxmax(), note_action.idxmin()
print(f"Utilisateur {u_pour} (note moyenne des films d'action {note_action[u_pour]:.2f}) et {u_contre} ({note_action[u_contre]:.2f}) : "
      f"{len(set(listes_ub[u_pour]) & set(listes_ub[u_contre]))} films en commun dans leurs top 10 user-based")
pd.DataFrame({f"utilisateur {u_pour}": avec_genres(listes_ub[u_pour])["titre"].to_numpy(),
              f"utilisateur {u_contre}": avec_genres(listes_ub[u_contre])["titre"].to_numpy()})

**Réponse.** **Oui, mais moins qu'on ne le croirait.** Deux utilisateurs pris au hasard ont en moyenne 1,9 film
en commun dans leurs top 10, et 21 % des paires n'en ont aucun ; le top des ventes, identique pour tous à quelques
films vus près, en a 5,4. Mais 3,2 des 10 films d'une liste viennent du top des ventes : un tiers de chaque liste est
commun à tout le monde, les deux autres tiers sont propres à l'utilisateur. L'item-based est plus personnalisé
(1,1 film en commun, 43 % des paires sans aucun).

L'exemple le confirme : l'utilisateur 298, qui adore les films d'action (note moyenne 4,6), et le 457, qui les déteste
(1,9), n'ont que 2 films en commun. Le premier reçoit *Fight Club*, *Léon*, *Star Wars* ; le second *Le Parrain*,
*Monty Python*, *Memento*, *Alien*. Les listes suivent les goûts, dans les limites d'un réservoir de films populaires.


### 5) Comment améliorer le système en production ?

**Réponse.** Dans l'ordre de ce que les mesures du TP justifient.

1. **Changer de système, ou combiner.** L'item-based avec classement par somme fait mieux sur tout : NDCG 0,249 contre
   0,191, RMSE 0,915 contre 0,999, films moins populaires et 410 films distincts contre 249 (exercice 4, bonus). Il est
   aussi plus explicable, « parce que vous avez aimé *Le Parrain* » (cours, page 26). Le user-based garde un intérêt :
   des listes différentes, donc un *hybride* par pondération ou par mélange des deux listes (cours, page 38) est la
   première chose à tester.
2. **Prédire les notes avec un modèle.** Si l'application affiche des notes prédites, le user-based brut est le pire
   choix (moins bon que la moyenne de l'utilisateur). Le centrage donne 0,932, l'item-based 0,915, et la
   *factorisation matricielle* du cours (page 44), qui apprend un vecteur par utilisateur et par film, fait mieux
   encore sur ce jeu de données d'après les résultats publiés. C'est la réponse à « comment intégrer du machine
   learning » (cours, page 19).
3. **Tempérer la popularité.** 97 % du catalogue n'est jamais proposé (question 3). Deux leviers simples : pénaliser le
   score des films très vus, ou réserver une partie de la liste à des films moins connus proposés par l'item-based,
   en mesurant la couverture et la diversité à côté du NDCG, comme on l'a fait ici.
4. **Le démarrage à froid.** Un nouvel utilisateur sans avis ou un nouveau film sans note ne peuvent rien recevoir
   du filtrage collaboratif. Le catalogue a des genres, des résumés, des acteurs : un filtrage par contenu (cours,
   page 28) prend le relais pour eux, en commutation (cours, page 38).
5. **Mesurer comme en vrai.** Notre découpage cache des avis au hasard ; en production on prédit l'avenir, donc il
   faudrait cacher les avis les plus récents (les timestamps existent). Et la seule mesure finale est le *test A/B*
   (cours, page 47) : une partie des utilisateurs reçoit le nouveau système, et on compare ce qu'ils regardent
   vraiment, y compris des signaux implicites (visionnage, temps passé) plutôt que des notes.
6. **Le passage à l'échelle.** Une matrice de similarité 671 × 671 se recalcule en une seconde ; avec des millions
   d'utilisateurs, il faut des matrices creuses, un recalcul périodique plutôt qu'à chaque requête, et des listes
   précalculées. L'item-based a l'avantage que la similarité entre films change lentement.
